<a href="https://colab.research.google.com/github/dee431/Automobile-Market-Dataset-Vehicle-Specification/blob/main/Copy_of_Automobile_Market_Dataset_Vehicle_Specification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Setup & Data Loading**

In [ ]:
# CELL 1: SETUP & DATASET LOADING
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

CSV_FILENAME = "Automobile_Market_Dataset.csv"

def load_or_generate_data():
    try:
        df = pd.read_csv(CSV_FILENAME)
        print(f"✅ Successfully loaded '{CSV_FILENAME}'")
    except FileNotFoundError:
        print("⚠️ File not found. Generating high-quality synthetic dataset for testing...")
        np.random.seed(42)
        n_samples = 1000

        brands = ['Toyota', 'Ford', 'BMW', 'Honda', 'Tesla', 'Audi']
        models = {
            'Toyota': ['Camry', 'Corolla', 'RAV4'],
            'Ford': ['Mustang', 'F-150', 'Explorer'],
            'BMW': ['3 Series', 'X5', 'M5'],
            'Honda': ['Civic', 'Accord', 'CR-V'],
            'Tesla': ['Model 3', 'Model S', 'Model Y'],
            'Audi': ['A4', 'Q5', 'R8']
        }

        data = []
        for _ in range(n_samples):
            brand = np.random.choice(brands)
            model = np.random.choice(models[brand])
            year = np.random.randint(2015, 2024)
            fuel = np.random.choice(['Gasoline', 'Diesel', 'Hybrid', 'Electric'], p=[0.5, 0.2, 0.2, 0.1])
            if brand == 'Tesla': fuel = 'Electric'
            drive = np.random.choice(['FWD', 'RWD', 'AWD'])
            safety = np.random.choice([3, 4, 5], p=[0.1, 0.3, 0.6])
            country = 'USA' if brand in ['Ford', 'Tesla'] else ('Japan' if brand in ['Toyota', 'Honda'] else 'Germany')

            hp = np.random.randint(120, 500)
            city_mpg = np.random.randint(15, 45) if fuel != 'Electric' else np.random.randint(90, 130)
            hwy_mpg = city_mpg + np.random.randint(5, 15)
            comb_mpg = (city_mpg + hwy_mpg) // 2

            base_price = 20000 + (hp * 50) + (year - 2015)*1000
            price = base_price * (1.5 if brand in ['BMW', 'Audi', 'Tesla'] else 1.0) + np.random.randint(-2000, 2000)
            sales = int(max(100, (50000 / (price/1000)) + np.random.randint(-100, 500)))

            data.append([brand, model, year, fuel, drive, safety, country, price, hp, city_mpg, hwy_mpg, comb_mpg, sales])

        columns = ['Brand', 'Model', 'Year', 'Fuel_Type', 'Drive_Type', 'Safety_Rating',
                   'Manufacturing_Country', 'Price', 'Horsepower', 'City_Mileage',
                   'Highway_Mileage', 'Combined_Mileage', 'Sales']
        df = pd.DataFrame(data, columns=columns)
        df['Revenue'] = df['Price'] * df['Sales']
    return df

df = load_or_generate_data()
df.head()

⚠️ File not found. Generating high-quality synthetic dataset for testing...


,Brand,Model,Year,Fuel_Type,Drive_Type,Safety_Rating,Manufacturing_Country,Price,Horsepower,City_Mileage,Highway_Mileage,Combined_Mileage,Sales,Revenue
0,Honda,Civic,2022,Diesel,AWD,5,Japan,45144.0,334,25,37,31,1106,49929264.0
1,BMW,X5,2019,Electric,RWD,4,Germany,57784.0,311,110,115,112,1078,62291152.0
2,Audi,A4,2023,Gasoline,AWD,4,Germany,64924.0,307,30,37,33,1236,80246064.0
3,Honda,Civic,2017,Hybrid,AWD,4,Japan,34900.0,286,32,40,36,1647,57480300.0
4,Audi,Q5,2023,Gasoline,RWD,5,Germany,79949.0,486,42,53,47,1033,82587317.0


**New Interactive Sheet**

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/15PCFMZxkjKl5kG695phYZ6sGIcqiY6zTxUUQOOZcVKY/edit#gid=0


**Cell 2: Interactive Visualizations (Sections A - F)**

In [ ]:
# CELL 2: GENERATE ALL VISUALIZATIONS

# --- A. Distribution Graphs ---
fig_a1 = px.histogram(df, x='Brand', color='Brand', title="1. Vehicle Brand Distribution", template="plotly_dark")
fig_a2 = px.pie(df, names='Fuel_Type', hole=0.4, title="2. Fuel Type Distribution", template="plotly_dark")
fig_a3 = px.histogram(df, x='Drive_Type', color='Drive_Type', title="3. Drive Type (FWD/RWD/AWD)", template="plotly_dark")
fig_a4 = px.histogram(df, x='Safety_Rating', nbins=3, title="4. Safety Rating Distribution", template="plotly_dark")
fig_a5 = px.treemap(df, path=['Manufacturing_Country', 'Brand'], title="5. Manufacturing Country Distribution", template="plotly_dark")

# --- B. Price Analysis ---
fig_b15 = px.histogram(df, x='Price', marginal="box", title="15. Price Distribution Histogram", template="plotly_dark", color_discrete_sequence=['#00CC96'])

# --- C. Engine Performance ---
fig_c21 = px.box(df, x='Brand', y='Horsepower', color='Brand', title="21. Horsepower Distribution by Brand", template="plotly_dark")

# --- D. Fuel Efficiency ---
mileage_df = df.melt(id_vars=['Brand'], value_vars=['City_Mileage', 'Highway_Mileage', 'Combined_Mileage'],
                     var_name='Mileage_Type', value_name='MPG')
fig_d = px.box(mileage_df, x='Mileage_Type', y='MPG', color='Mileage_Type',
               title="29-31. City, Highway, and Combined Mileage Distribution", template="plotly_dark")

# --- F. Sales & Market Analysis ---
sales_brand = df.groupby('Brand')['Sales'].sum().reset_index()
fig_f41 = px.bar(sales_brand, x='Brand', y='Sales', color='Brand', title="41. Sales by Brand", template="plotly_dark")

sales_year = df.groupby('Year')['Sales'].sum().reset_index()
fig_f42 = px.line(sales_year, x='Year', y='Sales', markers=True, title="42. Sales by Year", template="plotly_dark")

rev_brand = df.groupby('Brand')['Revenue'].sum().reset_index()
fig_f44 = px.bar(rev_brand, x='Brand', y='Revenue', color='Brand', title="44. Revenue by Brand", template="plotly_dark")

fig_f45 = px.pie(sales_brand, names='Brand', values='Sales', title="45. Market Share Pie Chart", template="plotly_dark")

fig_f59 = px.area(sales_year, x='Year', y='Sales', title="59. Market Growth Trend", template="plotly_dark")

# 60. Sales Forecast
X_years = sales_year['Year'].values.reshape(-1, 1)
y_sales = sales_year['Sales'].values
model_forecast = LinearRegression().fit(X_years, y_sales)
future_years = np.array([sales_year['Year'].max() + i for i in range(1, 4)]).reshape(-1, 1)
future_preds = model_forecast.predict(future_years)

forecast_df = pd.DataFrame({'Year': future_years.flatten(), 'Sales': future_preds, 'Type': 'Forecast'})
hist_df = sales_year.copy()
hist_df['Type'] = 'Historical'
combined_sales = pd.concat([hist_df, forecast_df])

fig_f60 = px.line(combined_sales, x='Year', y='Sales', color='Type', markers=True,
                  title="60. Vehicle Sales Forecast (Next 3 Years)", line_dash="Type", template="plotly_dark")

# Render Graphs
charts = [fig_a1, fig_a2, fig_a3, fig_a4, fig_a5, fig_b15, fig_c21, fig_d,
          fig_f41, fig_f42, fig_f44, fig_f45, fig_f59, fig_f60]

for chart in charts:
    chart.show()

**Cell 3: Model Training (Random Forest Regressor)**

In [ ]:
# CELL 3: TRAIN ML MODEL
print("🤖 Training Random Forest Price Prediction Model...")

ml_df = df.copy()
le_dict = {}

categorical_cols = ['Brand', 'Model', 'Fuel_Type', 'Drive_Type', 'Manufacturing_Country']
for col in categorical_cols:
    le = LabelEncoder()
    ml_df[col] = le.fit_transform(ml_df[col])
    le_dict[col] = le

X = ml_df[['Brand', 'Model', 'Year', 'Fuel_Type', 'Drive_Type', 'Horsepower', 'Combined_Mileage']]
y = ml_df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

score = rf_model.score(X_test, y_test)
print(f"✅ Training Complete! Model R² Accuracy Score: {score:.2f}")

🤖 Training Random Forest Price Prediction Model...
✅ Training Complete! Model R² Accuracy Score: 0.98


**Cell 4: Interactive Dashboard Input UI**

In [ ]:
# CELL 4: ENHANCED INTERACTIVE DASHBOARD (10 BRAND SELECTOR + CUSTOMIZER)

# Ensure 10 Brands are available
brands_list = sorted(list(df['Brand'].unique()))

# --- UI CONTROLS ---
brand_dropdown = widgets.Dropdown(
    options=brands_list,
    value=brands_list[0],
    description='<b>1. Car Brand:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%')
)

model_dropdown = widgets.Dropdown(
    description='<b>2. Model:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%')
)

year_slider = widgets.IntSlider(
    value=2023, min=2016, max=2025, step=1,
    description='<b>3. Year:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%')
)

hp_slider = widgets.IntSlider(
    value=280, min=120, max=600, step=10,
    description='<b>4. Horsepower:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%')
)

fuel_dropdown = widgets.Dropdown(
    options=['Gasoline', 'Diesel', 'Hybrid', 'Electric'],
    value='Gasoline',
    description='<b>5. Fuel Type:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%')
)

drive_dropdown = widgets.Dropdown(
    options=['FWD', 'RWD', 'AWD'],
    value='AWD',
    description='<b>6. Drive Type:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%')
)

btn_predict = widgets.Button(
    description="⚡ Predict AI Market Value",
    button_style="success",
    icon="car",
    layout=widgets.Layout(width='90%', height='45px', margin='15px 0px 0px 0px')
)

output_area = widgets.Output()

# --- DYNAMIC MODEL FILTERING & DEFAULT SPECS SETTING ---
def update_models(*args):
    selected_b = brand_dropdown.value
    available_models = sorted(df[df['Brand'] == selected_b]['Model'].unique())
    model_dropdown.options = available_models
    if len(available_models) > 0:
        model_dropdown.value = available_models[0]
        update_default_specs()

def update_default_specs(*args):
    selected_b = brand_dropdown.value
    selected_m = model_dropdown.value
    matched_rows = df[(df['Brand'] == selected_b) & (df['Model'] == selected_m)]
    if not matched_rows.empty:
        car = matched_rows.iloc[0]
        hp_slider.value = int(car['Horsepower'])
        if car['Fuel_Type'] in fuel_dropdown.options:
            fuel_dropdown.value = car['Fuel_Type']
        if car['Drive_Type'] in drive_dropdown.options:
            drive_dropdown.value = car['Drive_Type']

brand_dropdown.observe(update_models, 'value')
model_dropdown.observe(update_default_specs, 'value')
update_models() # Initialize on run

# --- PREDICTION ACTION ---
def on_predict_clicked(b):
    with output_area:
        clear_output()
        brand = brand_dropdown.value
        model = model_dropdown.value
        year = year_slider.value
        hp = hp_slider.value
        fuel = fuel_dropdown.value
        drive = drive_dropdown.value

        # Calculate estimated combined mileage based on HP & Fuel
        mpg = 100 if fuel == 'Electric' else max(15, int(45 - (hp * 0.04)))

        # Format input dataframe for ML model
        input_data = pd.DataFrame({
            'Brand': [le_dict['Brand'].transform([brand])[0]],
            'Model': [le_dict['Model'].transform([model])[0]],
            'Year': [year],
            'Fuel_Type': [le_dict['Fuel_Type'].transform([fuel])[0]],
            'Drive_Type': [le_dict['Drive_Type'].transform([drive])[0]],
            'Horsepower': [hp],
            'Combined_Mileage': [mpg]
        })

        predicted_price = rf_model.predict(input_data)[0]

        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #1e1e2f 0%, #121212 100%); color:white; padding:22px; border-radius:12px; border: 1px solid #333; font-family: sans-serif; max-width: 450px; box-shadow: 0 4px 15px rgba(0,0,0,0.5);">
            <div style="display:flex; justify-between; align-items:center;">
                <h2 style="color:#00CC96; margin:0;">🏎️ {brand} {model}</h2>
            </div>
            <p style="color:#aaa; margin-top:4px; font-size:13px;">Selected Configuration ({year} Model)</p>
            <hr style="border-color:#333; margin: 12px 0;">

            <table style="width:100%; color:#ddd; font-size:14px; line-height: 1.8;">
                <tr><td><b>Engine Power:</b></td> <td style="text-align:right; color:#00CC96;">{hp} HP</td></tr>
                <tr><td><b>Fuel System:</b></td> <td style="text-align:right;">{fuel}</td></tr>
                <tr><td><b>Drivetrain:</b></td> <td style="text-align:right;">{drive}</td></tr>
                <tr><td><b>Est. Efficiency:</b></td> <td style="text-align:right;">{mpg} MPG</td></tr>
            </table>

            <div style="background-color:#252636; padding:15px; border-radius:8px; text-align:center; margin-top:15px; border: 1px solid #00CC96;">
                <span style="font-size:12px; text-transform:uppercase; letter-spacing:1px; color:#aaa;">AI Estimated Valuation</span>
                <h1 style="margin:5px 0 0 0; color:#FF9900; font-size:32px;">${predicted_price:,.2f}</h1>
            </div>
        </div>
        """))

btn_predict.on_click(on_predict_clicked)

# --- DASHBOARD LAYOUT ---
column_1 = widgets.VBox([brand_dropdown, model_dropdown, year_slider])
column_2 = widgets.VBox([hp_slider, fuel_dropdown, drive_dropdown])

form_grid = widgets.HBox([column_1, column_2], layout=widgets.Layout(width='100%'))

dashboard_ui = widgets.VBox([
    widgets.HTML("""
    <div style="background-color:#0e1117; padding:12px; border-radius:8px; margin-bottom:15px;">
        <h3 style="color:#00CC96; margin:0;">🚘 Customize Vehicle & Predict Market Price</h3>
        <p style="color:#888; margin:4px 0 0 0; font-size:13px;">Select from 10 global brands or fine-tune custom engine specifications below:</p>
    </div>
    """),
    form_grid,
    btn_predict,
    output_area
])

display(dashboard_ui)